# You can run sapphire interactively in 3 simple steps:
1. import sapphire
2. define python dictionary of input parameters (see docs and below) 
3. sapphire.run(parameters) 

Soon there will be a way to run from the command line "python -m sapphire parameters.json" 
(useful if you want to run many parameter variations simultaneously on different nodes) 

In [9]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from timeit import default_timer as timer
import pandas as pd
from astropy.table import Table
from astropy.cosmology import Planck15

import sapphire 

In [10]:
""" This is an optional feature we are using to downsample the # of halos we will model to speed up computation time """

# define a list of (Mlow,Mhigh,N) tuples with same bin definitions as Behroozi+19 relation
# we will only model N random halos within each of these z=0 Mvir mass bins to cut down on modeling time
# this list of tuples is an input runtime parameter for sapphire below (along with np.random.seed(#) for reproducibility) 
downsample_bins = np.arange(10.0,12.4,0.2) # note the masses are log10 mvir
print(downsample_bins)

downsample_defs = [(downsample_bins[i],downsample_bins[i+1],5) for i in range(len(downsample_bins)-1)]

downsample_defs

[10.  10.2 10.4 10.6 10.8 11.  11.2 11.4 11.6 11.8 12.  12.2 12.4]


[(10.0, 10.2, 5),
 (10.2, 10.399999999999999, 5),
 (10.399999999999999, 10.599999999999998, 5),
 (10.599999999999998, 10.799999999999997, 5),
 (10.799999999999997, 10.999999999999996, 5),
 (10.999999999999996, 11.199999999999996, 5),
 (11.199999999999996, 11.399999999999995, 5),
 (11.399999999999995, 11.599999999999994, 5),
 (11.599999999999994, 11.799999999999994, 5),
 (11.799999999999994, 11.999999999999993, 5),
 (11.999999999999993, 12.199999999999992, 5),
 (12.199999999999992, 12.399999999999991, 5)]

In [11]:
# set up a general parameters dict 

# try running new thermal_general model on same 656 TNG trees 
parameters = {'output_path':'/Users/viraj/sapphire/demo/', # path where any output files should be saved (irrelevant if return_or_write=='return' below)
              
              ### runtime-related related parameters (you only need to vary tree_path) 
              
              'tree_type':'tng_sapphire', # type of merger trees / MAHs -- this determines what read_trees module gets loaded
              'tree_path':'/Users/viraj/sapphire/data/tng_trees_sapphire.npz', # either directory containing tree files or path to single tree file itself in case of TNG experimentation
              'halo_names':[], # optional for certain simulations -- list of halo names/IDs to process (default=empty=process all trees for given tree_type and tree_dir)
              'subvolumes':None, # for testing speed purposes, only work with trees from a single TNG subvolume 
              'num_readers':5, # irrelevant for tree_type='tng_sapphire'
              'num_writers':5, # irrelevant when return_or_write = 'return' (otherwise how many output writer tasks to start in parallel)
              'return_or_write':'return', # if 'return', return pandas dataframe (you should assign to a variable below); if 'write' then save dataframe to hdf5
              'min_root_mass':1e10, # relevant for tree_type ='tng' and 'tng_sapphire', not 'fire2' or 'fire2_pandya22'
              'max_root_mass':3e12, # relevant for tree_type=tng_sapphire -- halos above this z=0 Mvir will not be modeled (set to None or 0 if you don't want)
              'downsample_defs':downsample_defs, # list of tuples (logMlow,logMhigh,N) such that N random halos will be chosen in each mass bin (None otherwise)
              'downsample_seed':999, # np.random.seed(#) to ensure reproducibility when downsampling to N random halos in each mass bin (None otherwise)
              'rtol':1e-5, # relative error tolerance for ODE solver (this controls error on exponent of state variables), should be 1e-3 or lower 
              'atol':1e-3, # absolute error tolerance for ODE solver (this controls early evolution when ICs close to 0), should be 1e-3 or lower 
              
              ### fixed physical model parameters (this just sets the overall physical model and associated fixed parameters) 
              
              'physical_model':'thermal_general', # name of physical model (and associated free parameter functional forms)
              'coolfunc':'wiersma09', # which cooling function to use (wiersma09, sd93, ploeckinger20)
              'alpha_n':-3/2., # slope of CGM density power law
              'alpha_T':0.0, # slope of CGM temperature power law 
              'tau_escape':1.0, # scales the halo outflow timescale tdyn=tau*Rvir/Vvir
              'f_recycle':0.4, # instantaneous stellar-->ISM recycling fraction 
              'e_wind_halo':1.0, # specific energy of halo wind relative to Ecgm/Mcgm                            
              'yZ':0.02, # metal yield of 1 SN per 100 Msun of stars formed (2 Msun / 100 Msun = 0.02 for 10 Msun of SN ejecta)
              'return_all':False, # whether to return only the ODE RHS for solver, or all properties at each time for outputting
              
              ### FREE PARAMETERS BELOW THIS LINE -- WE WILL VARY A SUBSET OF THESE 
              ### there are 4 physical parameters: mass, energy and metal loading factors, and the ISM depletion time
              ### each physical parameter has 4 hyperparameters describing a power law: normalization, slope, and the redshift dependences of those two
              ### the exact power law is X(Vvir,z) = A * (Vvir/125.)**(alpha0 + alphaz*(1+z)) * (1+z)**beta              
              
              'etaM_A':1.0, # normalization of power law for mass loading factor [prior range: 0.01 to 100]
              'etaM_alpha0':-0.5, # slope of power law for mass loading factor [prior range: -2 to 2]
              'etaM_alphaz':0.0, # this can be fixed to 0; redshift dependence of slope of power law for mass loading factor [prior range: -2 to 2]
              'etaM_beta':0.0, # this can be fixed to 0; redshift dependence of normalization of power law for mass loading factor [prior range: -2 to 2]
              'tdep_A':3.0, # [prior range: 0.01 to 10]
              'tdep_alpha0':-3.0, # [prior range: -2 to 2]
              'tdep_alphaz':0.0, # this can be fixed to 0 [prior range: -2 to 2]
              'tdep_beta':-0.7, # [prior range: -2 to 2]
              'etaE_A':0.1, # [prior range: 0.01 to 1]
              'etaE_alpha0':-0.5, # [prior range: -2 to 2]
              'etaE_alphaz':0.0, # this can be fixed to 0 [prior range: -2 to 2]
              'etaE_beta':0.0,   # this can be fixed to 0 [prior range: -2 to 2]
              'etaZ_A':0.5, # [prior range: 0.01 to 1]
              'etaZ_alpha0':0.0, # [prior range: -2 to 2]
              'etaZ_alphaz':0.0, # this can be fixed to 0 [prior range: -2 to 2]
              'etaZ_beta':0.0} # this can be fixed to 0 [prior range: -2 to 2]



In [12]:
tstart = timer()
df_results = sapphire.run(parameters) # this is the only line you need, I just enclosed it in a timer for benchmarking
print('Finished in %s sec'%(timer()-tstart))


Parsing input parameters...
Reading trees...
Interpolating trees...
Evolving model halos...
We will process 10 trees in parallel...
Returning dictionary of results directly instead of writing output files
Finished in 27.101439082995057 sec


In [13]:
df_results

,Mdot_ism,etaM_ism,Mdot_in_halo,MZdot_cgm,Zin_halo,MZdot_cool,Delta_Ecgm,MZdot_in_halo,Edot_cgm_th,Vvir,...,MZdot_wind,MZdot_sfr,f_prev,redshift,Mfilt,sol_nfev,sol_njev,sol_nlu,sol_success,sol_status
1124053897,"[-0.023491454924721596, -0.023544219300347117,...","[2.247896259240123, 2.247605854595712, 2.24731...","[0.0339966607895436, 0.034047019786033327, 0.0...","[3.0927039723373526, 3.0879022559433578, 3.083...","[9.134509146634161e-06, 9.163024107949978e-06,...","[0.0, 0.0, 0.0, 2.369882637939498e-07, 2.45204...","[-3.3513568835784095e+53, -3.353052462237614e+...","[3.10542808937105e-07, 3.119736631032733e-07, ...","[321.60445632969487, 165.34724325644325, 1.752...","[24.737595499059324, 24.743988414180876, 24.75...",...,"[1.1309365865462254e-13, 1.1302450417755032e-1...","[1.3565140432867684e-17, 1.5203175281706566e-1...","[0.2718815021572879, 0.2718984914763918, 0.271...","[7.23655414581297, 7.220424152282052, 7.204325...","[42866906.42878141, 43135188.49679358, 4340287...","[1221.0, 1221.0, 1221.0, 1221.0, 1221.0, 1221....","[63.0, 63.0, 63.0, 63.0, 63.0, 63.0, 63.0, 63....","[247.0, 247.0, 247.0, 247.0, 247.0, 247.0, 247...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1124054144,"[-0.03780458613974653, -0.03791539076113144, 1...","[2.110935696725963, 2.1103268892128777, 2.1097...","[0.0942491049547771, 0.09423347743798831, 0.09...","[10.006626699885475, 9.80536791258859, 7.02925...","[1.531689301633218e-05, 1.537182848867575e-05,...","[0.0, 0.0, 4.173086743898142e-07, 4.8393829279...","[-8.172881788681012e+53, -4.691262291049416e+5...","[1.4436034574773841e-06, 1.4485408530682523e-0...","[738.9295776852532, 147.47255632639556, -0.596...","[28.051752916849974, 28.06794053351603, 28.084...",...,"[1.4489057121826123e-13, 1.4495778607054949e-1...","[1.7379531131659715e-17, 2.0621848684966803e-1...","[0.281840167282988, 0.2818945904028991, 0.2819...","[5.846501350402847, 5.834341159172865, 5.82220...","[65651560.14734047, 65849503.3398703, 66047147...","[955.0, 955.0, 955.0, 955.0, 955.0, 955.0, 955...","[54.0, 54.0, 54.0, 54.0, 54.0, 54.0, 54.0, 54....","[164.0, 164.0, 164.0, 164.0, 164.0, 164.0, 164...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1124055085,"[-0.02552709748493306, -0.02562660657758727, 3...","[2.181945790641873, 2.181005844651545, 2.18006...","[0.08322393522094174, 0.08338748022949888, 0.0...","[6.37904326259853, 6.315982005215431, 2.666672...","[9.263454675015456e-06, 9.305792909641862e-06,...","[0.0, 0.0, 5.131806333773884e-07, 5.2817652607...","[-3.970236534321834e+53, -2.189674305241756e+5...","[7.709411517956162e-07, 7.759866222725715e-07,...","[710.240045688371, 3.1573613269982577, -2.2519...","[26.25560922228359, 26.278244808138403, 26.300...",...,"[1.4400844827432436e-13, 1.411586531397227e-13...","[1.7273475835341192e-17, 1.932678665343999e-17...","[0.27615367351928033, 0.2762210354101969, 0.27...","[8.01225662231446, 7.993806021756151, 7.975392...","[29945012.95596898, 30250826.81352764, 3055620...","[1034.0, 1034.0, 1034.0, 1034.0, 1034.0, 1034....","[60.0, 60.0, 60.0, 60.0, 60.0, 60.0, 60.0, 60....","[221.0, 221.0, 221.0, 221.0, 221.0, 221.0, 221...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1124013691,"[-0.0513464062645911, 802.7923776115547, 228.1...","[1.8951121045937422, 1.8942557265625797, 1.893...","[0.3296356080396484, 0.3300168237076777, 0.330...","[18.880292142858682, 12.385000825926342, 13.62...","[1.6194511149368665e-05, 1.6271029738535054e-0...","[0.0, 1.9349409631155975e-06, 2.04098169077954...","[-1.5211770614821209e+54, -1.1374889117491747e...","[5.338287529627005e-06, 5.3697135527645045e-06...","[1790.7476027008665, -3.6573383846064904, -3.7...","[34.804884897219104, 34.83636203723233, 34.867...",...,"[3.4674302772765076e-13, 2.3673134711015914e-0...","[4.159339849672563e-17, 3.143524954288952e-11,.

In [14]:
# use list comprehension to store final z=0 Mvir and Mstar of each halo (i.e., of each row aka roothaloid of df_filtered)
# here we are iterating over each row which is a distinct roothaloid
# the values of every column for a given row/roothaloid are numpy arrays of time series of that property going from high-z to z~0

z0_Mvir = np.array([df_results.iloc[irow]['Mvir'][-1] for irow in range(len(df_results))])
z0_Mstar = np.array([df_results.iloc[irow]['M_star'][-1] for irow in range(len(df_results))])

In [ ]:
# read in Behroozi+19 UniverseMachine DR1 Median SMHM relation at z=0  
# and discard bins below logmvir<10.5 due to Bolshoi-Planck resolution limit (not sure why he included these)
tum = Table.read('smhm_a1.002312.dat',format='ascii')
tum = tum[(tum['HM(0)']>=10.0) & (tum['HM(0)']<=12.4)]


In [ ]:
# compare SMHM relation from Behroozi vs this realization of our model

fig, ax = plt.subplots(1,figsize=(4,4),dpi=150)

# plot Behroozi+19 relation including only central SF galaxies/halos
ax.errorbar(10**tum['HM(0)'],tum['Med_Cen_SF(7)'],yerr=[np.abs(tum['Err-(9)']),np.abs(tum['Err+(8)'])],fmt='b-',capsize=3,label='Behroozi+19 (SF Centrals)')

ax.plot(z0_Mvir,np.log10(z0_Mstar/z0_Mvir),'k.',alpha=0.5,mec='none',label='sapphire realization')

ax.set_xscale('log')
ax.set_xlim(None,1e13)
ax.set_ylim(-4,-0.3)

ax.axhline(np.log10(Planck15.Ob0/Planck15.Om0),color='r',ls=':',alpha=0.5,label=r'cosmic $f_b\approx0.16$')

ax.legend(loc='lower right',fontsize=8,fancybox=True,framealpha=0)

ax.set_xlabel(r'$M_{\rm vir}$ [M$_{\odot}$] (z=0)',fontsize=12)
ax.set_ylabel(r'$\log_{10}M_{\rm star}/M_{\rm vir}$ (z=0)',fontsize=12)


In [ ]:
### now what if we wanted to do another realization with updated parameters?
### in that case, we just update the parameters dictionary and re-call sapphire.run 

# for example here we increase feedback via increasing specific energy of galactic winds
# i.e., we decrease the mass loading factor normalization and increase the energy loading factor normalization, with all else fixed
parameters.update({'etaM_A':0.1, 'etaE_A':1.0}) # before these were 1.0 and 0.1 respectively 

In [ ]:
tstart = timer()
df_results2 = sapphire.run(parameters) # assign to a new dataframe variable name 
print('Finished in %s sec'%(timer()-tstart))


In [ ]:
# re-pull the new z=0 Mvir and Mstar values
z0_Mvir2 = np.array([df_results2.iloc[irow]['Mvir'][-1] for irow in range(len(df_results2))])
z0_Mstar2 = np.array([df_results2.iloc[irow]['M_star'][-1] for irow in range(len(df_results2))])

In [ ]:
# remake the previous plot with both model predictions now
# the new realization's SMHM points should be lower due to stronger feedback reducing z=0 Mstar 

fig, ax = plt.subplots(1,figsize=(4,4),dpi=150)

# plot Behroozi+19 relation including only central SF galaxies/halos
ax.errorbar(10**tum['HM(0)'],tum['Med_Cen_SF(7)'],yerr=[np.abs(tum['Err-(9)']),np.abs(tum['Err+(8)'])],fmt='b-',capsize=3,label='Behroozi+19 (SF Centrals)')

ax.plot(z0_Mvir,np.log10(z0_Mstar/z0_Mvir),'k.',alpha=0.5,mec='none',label='sapphire (weaker feedback)')
ax.plot(z0_Mvir2,np.log10(z0_Mstar2/z0_Mvir2),'m.',alpha=0.5,mec='none',label='sapphire (stronger feedback)')

ax.set_xscale('log')
ax.set_xlim(None,1e13)
ax.set_ylim(-4,-0.3)

ax.axhline(np.log10(Planck15.Ob0/Planck15.Om0),color='r',ls=':',alpha=0.5,label=r'cosmic $f_b\approx0.16$')

ax.legend(loc='lower right',fontsize=8,fancybox=True,framealpha=0)

ax.set_xlabel(r'$M_{\rm vir}$ [M$_{\odot}$] (z=0)',fontsize=12)
ax.set_ylabel(r'$\log_{10}M_{\rm star}/M_{\rm vir}$ (z=0)',fontsize=12)


In [ ]:
""" 
Suppose you wanted to compute the goodness of fit 
One way is to first bin up the SAM halos in the same mass bins as Behroozi and then compute the median SMHM of all halos in each mass bin

NOTE: I am not sure this is the best way to compare the model to the data
It is probably better to compute the deviation of each individual halo from the Behroozi+19 median for the mass bin it belongs to
But for that, I need to fix a minor bug with the interpolator inside sapphire for df_results['Mvir'], so maybe use this median approach for now ...
"""

def median_smhm(df,Mmin,Mmax):
    """ 
    Filters input dataframe to only consider halos with z=0 Mvir between Mmin and Mmax
    Then returns median SMHM ratio for those halos
    """ 
    
    Mvir_bin = np.array([df.iloc[irow]['Mvir'][-1] for irow in range(len(df)) 
                            if df.iloc[irow]['Mvir'][-1]>=10**Mmin and df.iloc[irow]['Mvir'][-1]<=10**Mmax])
    Mstar_bin = np.array([df.iloc[irow]['M_star'][-1] for irow in range(len(df)) 
                             if df.iloc[irow]['Mvir'][-1]>=10**Mmin and df.iloc[irow]['Mvir'][-1]<=10**Mmax])    
    
    return np.median(Mstar_bin / Mvir_bin)

# we will store median smhm of predicted halos in the two models above
median_smhm_weakfb = np.array([])
median_smhm_strongfb = np.array([]) 

for Mmin,Mmax,N in downsample_defs: # our Mmin,Mmax bin edges for downsampling the # of halos was the same as the Behroozi bin definitions

    ### filter dataframe for both models to only contain halos that fall within current z=0 Mvir bin

    # first the weak feedback model (df_results)
    median_smhm_weakfb = np.append(median_smhm_weakfb, median_smhm(df_results,Mmin,Mmax))
    
    # now the strong feedback model (df_results2)
    median_smhm_strongfb = np.append(median_smhm_strongfb, median_smhm(df_results2,Mmin,Mmax))
    

In [ ]:
### finally overplot the 2 median relations assuming the same bin centers as Behroozi+19 

fig, ax = plt.subplots(1,figsize=(4,4),dpi=150)

# plot Behroozi+19 relation including only central SF galaxies/halos
ax.errorbar(10**tum['HM(0)'],tum['Med_Cen_SF(7)'],yerr=[np.abs(tum['Err-(9)']),np.abs(tum['Err+(8)'])],fmt='b-',capsize=3,label='Behroozi+19 (SF Centrals)')

ax.plot(10**tum['HM(0)'],np.log10(median_smhm_weakfb),'ko-',alpha=0.5,mec='none',label='sapphire (weaker feedback)')
ax.plot(10**tum['HM(0)'],np.log10(median_smhm_strongfb),'mo-',alpha=0.5,mec='none',label='sapphire (stronger feedback)')

ax.set_xscale('log')
ax.set_xlim(None,1e13)
ax.set_ylim(-4,-0.3)

ax.axhline(np.log10(Planck15.Ob0/Planck15.Om0),color='r',ls=':',alpha=0.5,label=r'cosmic $f_b\approx0.16$')

ax.legend(loc='lower right',fontsize=8,fancybox=True,framealpha=0)

ax.set_xlabel(r'$M_{\rm vir}$ [M$_{\odot}$] (z=0)',fontsize=12)
ax.set_ylabel(r'$\log_{10}M_{\rm star}/M_{\rm vir}$ (z=0)',fontsize=12)
